In [2]:
# %% [markdown]
# # Task #1: Standardize Categorical Variables, Normalize Numerical Features
# Done by: Johnny Aryeetey
#
# Uses only the 10 listing-time features from DATA_DICTIONARY.md (no leaky columns).

# %%
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# %% [markdown]
# ## 1. Load data
# DUA-governed data: load from your LOCAL copy. Do NOT commit the CSV.
# Update DATA_PATH to wherever your team says the extract lives.

# %%
import os

LOCAL_PATH = "data/kidney_waitlist_analytic.csv.gz"  # works when run from a cloned repo root
RAW_URL = ("https://raw.githubusercontent.com/Break-Through-Tech/"
           "Healthcare-1B-predicting-organ-transplant-waitlist-outcomes-and-optimizing-allocation/"
           "main/data/kidney_waitlist_analytic.csv.gz")  # works in Colab

DATA_PATH = LOCAL_PATH if os.path.exists(LOCAL_PATH) else RAW_URL
df = pd.read_csv(DATA_PATH, compression="gzip", low_memory=False)
print(df.shape)

# %% [markdown]
# ## 2. Apply the team's cleaning decisions (from Task #3 notes)
# - Drop columns with 90%+ missingness
# - Null out impossible BMI values (<10 or >100) instead of dropping rows

# %%
df = df.drop(columns=[c for c in ["MULTIORG", "A2A2B_ELIGIBILITY"] if c in df.columns])

bmi_outliers = (df["BMI_TCR"] < 10) | (df["BMI_TCR"] > 100)
df.loc[bmi_outliers, "BMI_TCR"] = np.nan
print("BMI outliers nulled:", bmi_outliers.sum())

# %% [markdown]
# ## 3. Features
# FUNC_STAT_TCR has two coding scales (2010-2100 and 4010-4100) plus unknowns
# (996/998) and one rogue value (1). Placeholder harmonization below: map the
# 4000-series onto the 2000-series and treat 996/998/1 as missing.
# TEAM DECISION NEEDED: confirm this mapping (or encode the scale separately).

# %%
def harmonize_func_stat(s):
    s = s.copy()
    s = s.where(~s.isin([1, 996, 998]), np.nan)
    s = s.where(~s.between(4010, 4100), s - 2000)
    return s

df["FUNC_STAT_TCR"] = harmonize_func_stat(df["FUNC_STAT_TCR"])

categorical_features = ["GENDER", "ETHCAT", "ABO", "ON_DIALYSIS",
                        "FUNC_STAT_TCR", "INIT_STAT", "REGION"]
numerical_features = ["INIT_AGE", "INIT_CPRA", "BMI_TCR"]

# OneHotEncoder needs consistent types for mixed numeric-code columns
X = df[categorical_features + numerical_features].copy()
X[categorical_features] = X[categorical_features].astype("object")
X[categorical_features] = X[categorical_features].where(X[categorical_features].notna(), np.nan)

# %% [markdown]
# ## 4. Preprocessing pipeline
# Impute first, then encode/scale, so the output has no NaNs.

# %%
categorical_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

numerical_pipe = Pipeline([
    # add_indicator keeps a 0/1 "was missing" column; INIT_CPRA is ~29% missing
    ("impute", SimpleImputer(strategy="median", add_indicator=True)),
    ("scale", StandardScaler()),
])

preprocessor = ColumnTransformer([
    ("cat", categorical_pipe, categorical_features),
    ("num", numerical_pipe, numerical_features),
])

X_transformed = preprocessor.fit_transform(X)
print("Before:", X.shape)
print("After:", X_transformed.shape)

# %% [markdown]
# ## 5. Sanity check: no NaNs should remain

# %%
arr = X_transformed.toarray() if hasattr(X_transformed, "toarray") else X_transformed
print("NaNs remaining:", int(np.isnan(arr).sum()))

# NOTE: for real modeling, fit the preprocessor on the TRAIN split only
# (grouped by PT_CODE) and only .transform() the test split, to avoid leakage.

(494862, 38)
BMI outliers nulled: 503
Before: (494862, 10)
After: (494862, 51)
NaNs remaining: 0
